# 03 - PySpark ML: predykcja zużycia opon

Trening modelu regresyjnego (RandomForestRegressor) przewidującego `tire_wear_pct` na podstawie cech telemetrii. Ten notebook jest interaktywnym odpowiednikiem `spark_jobs/ml_training_job.py` - przydatny do eksperymentów i strojenia hiperparametrów.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("f1-ml-training-notebook")
    .master("local[*]")
    .getOrCreate()
)
spark

In [ ]:
# Ta sama sciezka co w kontenerach Sparka: _spark_metadata ze streamingu zawiera
# absolutne sciezki /opt/data/..., a Spark czyta tylko pliki pod podanym katalogiem.
INPUT_DIR = "/opt/data/processed/telemetry_agg"

df = spark.read.parquet(INPUT_DIR)
df.printSchema()
df.show(5)

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

FEATURE_COLS = [
    "speed_kmh", "throttle_pct", "brake_pct", "gear", "rpm", "lap", "distance_m",
    "tire_age_laps",
]
LABEL_COL = "tire_wear_pct"

clean_df = df.dropna(subset=FEATURE_COLS + [LABEL_COL])

compound_indexer = StringIndexer(
    inputCol="tire_compound", outputCol="tire_compound_idx", handleInvalid="keep"
)
assembler = VectorAssembler(
    inputCols=FEATURE_COLS + ["tire_compound_idx"], outputCol="features"
)
rf = RandomForestRegressor(
    featuresCol="features", labelCol=LABEL_COL, numTrees=100, maxDepth=8
)

pipeline = Pipeline(stages=[compound_indexer, assembler, rf])

In [ ]:
train_df, test_df = clean_df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_df)
predictions = model.transform(test_df)

evaluator = RegressionEvaluator(
    labelCol=LABEL_COL, predictionCol="prediction", metricName="rmse"
)
rmse = evaluator.evaluate(predictions)
print(f"Test RMSE: {rmse:.3f}")

predictions.select(LABEL_COL, "prediction").show(10)

In [ ]:
rf_model = model.stages[-1]
list(zip(FEATURE_COLS + ["tire_compound_idx"], rf_model.featureImportances.toArray()))

In [ ]:
MODEL_DIR = "/home/jovyan/data/models/tire_wear_rf"
model.write().overwrite().save(MODEL_DIR)
print(f"Model zapisany w {MODEL_DIR}")